In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd().parent

DATA_PATH = BASE_DIR / "data" / "ai4i2020.csv"
MODEL_PATH = BASE_DIR / "models" / "random_forest_pipeline.joblib"

FAILURE_THRESHOLD = 0.40

features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

df = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)

X = df[features]
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset loaded:", df.shape)
print("Test observations:", len(X_test))
print("Model loaded successfully.")

Dataset loaded: (10000, 14)
Test observations: 2000
Model loaded successfully.


In [2]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= FAILURE_THRESHOLD).astype(int)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Decision Threshold: {FAILURE_THRESHOLD:.0%}")
print(f"Precision:          {precision:.1%}")
print(f"Recall:             {recall:.1%}")
print(f"F1 Score:           {f1:.1%}")
print(f"ROC-AUC:            {roc_auc:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Decision Threshold: 40%
Precision:          61.2%
Recall:             76.5%
F1 Score:           68.0%
ROC-AUC:            0.963

Confusion Matrix:
[[1899   33]
 [  16   52]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1932
           1       0.61      0.76      0.68        68

    accuracy                           0.98      2000
   macro avg       0.80      0.87      0.83      2000
weighted avg       0.98      0.98      0.98      2000



In [3]:
test_cases = X_test.copy()

test_cases["Actual Failure"] = y_test
test_cases["Failure Probability"] = y_prob
test_cases["Predicted Failure"] = y_pred

test_cases.sample(10, random_state=42)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Actual Failure,Failure Probability,Predicted Failure
4701,H,303.5,311.7,1584,32.7,143,0,0.00,0
8311,M,298.8,310.0,1295,52.7,74,0,0.01,0
204,L,298.4,308.5,1741,32.8,112,0,0.00,0
4985,M,303.8,313.0,1789,23.5,215,0,0.11,0
6816,L,300.9,310.7,1477,43.2,143,0,0.00,0
3575,L,301.7,310.5,1567,39.1,114,0,0.00,0
2270,L,299.3,308.6,1437,48.1,67,0,0.01,0
1281,H,298.3,309.5,1427,54.3,89,0,0.01,0
4632,M,302.9,311.2,1330,53.7,200,1,0.44,1
4630,L,302.9,311.2,1751,23.8,196,0,0.00,0


In [4]:
test_cases[
    test_cases["Actual Failure"] == 1
].sort_values(
    "Failure Probability",
    ascending=False
).head(10)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Actual Failure,Failure Probability,Predicted Failure
1391,L,298.9,310.2,2737,8.8,142,1,0.93,1
4495,L,302.6,310.4,1359,57.2,67,1,0.92,1
7011,L,300.5,310.5,2633,12.2,17,1,0.91,1
380,L,297.5,308.3,2564,12.8,127,1,0.90,1
2444,L,299.2,308.6,1267,68.2,76,1,0.89,1
4661,L,303.3,311.3,1350,48.1,32,1,0.89,1
6078,L,300.8,310.7,1339,59.1,203,1,0.88,1
4427,M,302.4,310.1,1379,48.9,107,1,0.88,1
249,L,298.0,308.3,1405,56.2,218,1,0.87,1
7997,M,301.0,312.2,2710,9.7,143,1,0.86,1


In [5]:
test_cases[
    (test_cases["Actual Failure"] == 1) &
    (test_cases["Predicted Failure"] == 0)
].sort_values(
    "Failure Probability",
    ascending=False
).head(10)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Actual Failure,Failure Probability,Predicted Failure
3806,M,302.3,310.9,1360,44.0,67,1,0.35,0
4720,M,303.4,311.7,1353,44.1,192,1,0.34,0
4778,H,303.6,312.2,1371,54.6,112,1,0.30,0
926,L,295.6,306.1,1372,55.6,215,1,0.29,0
9015,L,297.2,308.1,1431,49.7,210,1,0.25,0
2864,H,300.6,309.4,1380,47.6,246,1,0.25,0
7884,L,300.8,312.4,1465,59.1,91,1,0.21,0
4034,L,302.0,310.8,1615,29.0,235,1,0.19,0
7849,M,300.3,311.7,1374,47.9,222,1,0.17,0
3695,L,302.2,311.3,1530,37.3,207,1,0.13,0


## Final Model Testing Results

The saved Random Forest model was successfully loaded and evaluated against
the held-out test dataset containing 2,000 observations. Using the 40%
classification threshold implemented in the application, the model achieved
61.2% precision, 76.5% recall, a 68.0% F1 score, and a ROC-AUC of 0.963.

The confusion matrix showed 52 correctly identified machine failures, 16
missed failures, 33 false-positive failure predictions, and 1,899 correctly
identified non-failure observations. Examination of individual test cases
also confirmed that the 40% decision threshold was applied correctly.

These results verify that the saved model produces the expected predictions
and performance metrics used by the final Streamlit application.

## Streamlit Application Testing

The final Streamlit application was manually tested using observations from
the held-out test dataset. The same feature values were entered through the
application's prediction interface and the resulting probabilities and
classifications were compared with the results produced during model testing.

Three representative cases were tested:

- **Known failure:** Test observation 1391 produced a 93.0% failure
  probability in both the test notebook and Streamlit application and was
  correctly classified as a potential machine failure.
- **Known non-failure:** Test observation 4701 produced a 0.0% failure
  probability in both environments and was correctly classified as no
  machine failure predicted.
- **Near-threshold failure:** Test observation 4632 produced a 44.0% failure
  probability in both environments. Because this exceeds the application's
  40% decision threshold, it was correctly classified as a potential machine
  failure.

The matching results confirm that the deployed application uses the saved
model and selected decision threshold consistently with the final model
evaluation.